# Урок 32 (Live Project) — Multithreaded News Scraper

## Переписуємо Sequential Scraper → ThreadPoolExecutor

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей notebook?

У Уроці 31 ми побудували scraper, що завантажує новини з rbc.ua **послідовно**:

```python
for url in PAGES_TO_SCRAPE:
    r = requests.get(url)   # Python зависає тут на ~1–3с
    news = parse_rbc_news(r.text)
```

**Проблема:** При 8 сторінках — `8 × 2с = 16с` очікування.  
Python **нічого не робить** поки чекає TCP відповідь.

**Рішення:** `ThreadPoolExecutor` — всі 8 запитів відправляємо **одночасно**.  
Реальний час: `max_latency` ≈ **~2–3с** незалежно від кількості сторінок.

---

# Чому threading тут ідеальний?

Web scraping — **IO-bound задача**:  
- CPU зайнятий < 1% часу (формування запиту + парсинг HTML)  
- 99% часу — очікування TCP відповіді від сервера

GIL **відпускається** під час `socket.recv()`.  
Поки Thread 1 чекає rbc.ua/news — Thread 2 вже надсилає запит на rbc.ua/economy.

---

# Що нового відносно Уроку 31?

| Компонент | Урок 31 | Цей notebook |
| --------- | ------- | ------------ |
| HTTP | `requests.get()` у циклі | `Session` + `ThreadPoolExecutor` |
| Порядок | Послідовний | Конкурентний (`as_completed`) |
| Парсинг | Монолітна функція | Чисті компонентні функції |
| Error handling | Базовий | Timeout + Retry + per-URL |
| Benchmark | Немає | Sequential vs Concurrent |


# Learning Objectives

Після цього notebook ти зможеш:

## Architecture

- Пояснити чому web scraping є **IO-bound** і чому GIL не перешкоджає
- Описати різницю між `executor.map` (порядок) та `as_completed` (швидкість)
- Пояснити чому `requests.Session` ефективніший за окремі `get()` виклики

## Engineering Skills

- Написати **thread-safe** дедуплікацію через `threading.Lock`
- Розбити монолітну parse-функцію на **чисті компоненти**
- Зробити benchmark sequential vs concurrent і проінтерпретувати результат

## Production Patterns

- Використовувати `future.result()` з try/except для silent failure detection
- Налаштувати `max_workers` оптимально для IO-bound задачі
- Побудувати повний pipeline: URL list → ThreadPool → parse → DataFrame


# 1. Архітектура: Threading Pipeline

## Flowchart: Sequential vs Concurrent

```mermaid
flowchart LR
    subgraph SEQ["❌ Sequential (Урок 31)"]
        direction TB
        S1["URL 1 → GET → parse"] --> S2["URL 2 → GET → parse"]
        S2 --> S3["URL 3 → GET → parse"]
        S3 --> S4["URL N → GET → parse"]
    end

    subgraph CONC["✅ Concurrent (Цей notebook)"]
        direction TB
        U["news_urls"] --> TPE["ThreadPoolExecutor"]
        TPE --> T1["Thread 1: GET url1"]
        TPE --> T2["Thread 2: GET url2"]
        TPE --> T3["Thread 3: GET url3"]
        TPE --> TN["Thread N: GET urlN"]
        T1 & T2 & T3 & TN --> BS["BeautifulSoup parse"]
        BS --> NL["news_list"]
    end
```

## Sequence: Що відбувається всередині

```mermaid
sequenceDiagram
    participant M as 🧵 Main Thread
    participant E as ⚙️ Executor
    participant T1 as 🧵 Thread 1
    participant T2 as 🧵 Thread 2
    participant S as 🌐 rbc.ua

    M->>E: submit(fetch_page, url1)
    M->>E: submit(fetch_page, url2)
    E->>T1: url1
    E->>T2: url2
    T1->>S: GET /
    T2->>S: GET /ukr/news/
    Note over T1,T2: Обидва чекають IO паралельно<br/>GIL відпущено під час socket.recv()
    S-->>T2: HTML (швидший)
    T2->>T2: parse_rbc_news()
    S-->>T1: HTML
    T1->>T1: parse_rbc_news()
    T1-->>M: Future result
    T2-->>M: Future result
```


In [ ]:
# === ІМПОРТИ ===
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from collections import Counter
import re
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm                             # progress bars
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

# tqdm.pandas() — вмикає .progress_apply() для pandas Series/DataFrame
tqdm.pandas(desc='Обробка')

print('✅ Всі бібліотеки імпортовано')
print(f'   requests: {requests.__version__}')
print(f'   pandas:   {pd.__version__}')
print(f'   tqdm:     {tqdm}')


# 2.5 tqdm — Progress Bar для Threading Pipeline

## Що показує tqdm

```
Парсимо RBC:  62%|████████████████▋         | 5/8 [00:03<00:02,  1.7it/s]
               ▲          ▲                    ▲      ▲        ▲
            відсоток   progress bar        кроки  elapsed  швидкість
```

## Навіщо tqdm у threading pipeline?

| Без tqdm | З tqdm |
| -------- | ------ |
| Не видно чи pipeline працює чи завис | Бачиш ETA та швидкість |
| Не видно bottleneck | Бачиш які URL повільні |
| При 1000+ URL — нічого | Бачиш `347/1000 [00:45<01:32, 7.2it/s]` |

## Як tqdm інтегрується в threading

```mermaid
flowchart LR
    FUT["as_completed(futures)"] -->|"кожен завершений Future"| TQDM
    subgraph TQDM["tqdm wrapper"]
        COUNT["лічильник ++"]
        BAR["оновлює progress bar"]
        ETA["рахує ETA"]
    end
    TQDM --> RES["result processing"]
```

## Три патерни використання

```python
# Патерн 1: простий for-loop (sequential)
for url in tqdm(urls, desc='Sequential'):
    fetch(url)

# Патерн 2: as_completed (concurrent) ← найкорисніший
for future in tqdm(as_completed(futures), total=len(futures), desc='Threads'):
    result = future.result()

# Патерн 3: pandas progress_apply
tqdm.pandas()
df['clean'] = df['title'].progress_apply(clean_text)
```


In [ ]:
# === tqdm DEMO: бачимо різницю Sequential vs Concurrent ===

import time
from tqdm import tqdm

DEMO_URLS = [f'item_{i}' for i in range(12)]

def fake_download(url: str, delay: float = 0.3) -> str:
    """Симуляція IO затримки."""
    time.sleep(delay)
    return f'data:{url}'

# ── Патерн 1: tqdm у for-loop ────────────────────────────────────────────────
print('Патерн 1: Sequential + tqdm')
results_seq = []
for url in tqdm(DEMO_URLS, desc='Sequential download', unit='url', colour='red'):
    results_seq.append(fake_download(url, delay=0.15))

print()

# ── Патерн 2: tqdm + as_completed ────────────────────────────────────────────
print('Патерн 2: Concurrent + tqdm (as_completed)')
results_conc = []

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(fake_download, url, 0.15) for url in DEMO_URLS]

    # tqdm обгортає as_completed — бачимо кожен завершений Future
    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc='Concurrent download',
        unit='url',
        colour='green'
    ):
        results_conc.append(future.result())

print(f'\n✅ Обидва зібрали {len(results_conc)} результатів')
print('Зверни увагу на it/s: concurrent значно вищий!')


In [ ]:
# === КОНФІГУРАЦІЯ ===

BASE_URL = 'https://www.rbc.ua'

# User-Agent — обов'язковий для ввічливого scraping
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; EduScraper/1.0; +educational)',
    'Accept-Language': 'uk,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml',
}

# Список сторінок для збору новин
# Більше URL → краще видно перевагу threading над sequential
PAGES_TO_SCRAPE = [
    'https://www.rbc.ua/',
    'https://www.rbc.ua/rus/news/',
    'https://www.rbc.ua/rus/economic/',
    'https://www.rbc.ua/rus/politics/',
    'https://www.rbc.ua/rus/society/',
    'https://www.rbc.ua/rus/sport/',
    'https://www.rbc.ua/rus/world/',
    'https://www.rbc.ua/rus/hitech',
]

REQUEST_TIMEOUT = 12   # секунди; якщо сервер не відповів — кидає Timeout
MAX_WORKERS     = 8    # один потік на URL — optimal для IO-bound задачі

print(f'Сторінок для збору: {len(PAGES_TO_SCRAPE)}')
print(f'Max workers:        {MAX_WORKERS}')
print(f'Timeout:            {REQUEST_TIMEOUT}с')


# 2. Чисті функції парсингу

## Принцип Single Responsibility

У Уроці 31 весь парсинг був в одній великій функції `parse_rbc_news()`.
Тут кожна функція відповідає за **одну атомарну операцію**:

```
find_news_containers()  → знаходить контейнери <div class='newsline__item'>
extract_title_tag()     → знаходить тег із заголовком
extract_title()         → витягує текст
extract_url()           → нормалізує відносний/абсолютний URL
extract_category()      → знаходить категорію
extract_description()   → витягує лід-абзац
extract_datetime()      → знаходить datetime
parse_news_item()       → збирає одну новину з контейнера
parse_links_fallback()  → fallback через усі <a href=/news/>
parse_rbc_news()        → головна функція: оркеструє всі вище
```

Перевага: кожну функцію можна **тестувати ізольовано** та **замінювати** незалежно.


In [ ]:
# === ЧИСТІ ФУНКЦІЇ ПАРСИНГУ ===


def find_news_containers(soup: BeautifulSoup) -> list:
    """Шукає основні контейнери новин за відомими CSS-класами."""
    item_selectors = [
        ('div',     'newsline__item'),
        ('li',      'newsline__item'),
        ('div',     'news-feed__item'),
        ('div',     'news-item'),
        ('article', 'news-item'),
        ('article', 'article-item'),
    ]
    for tag, cls in item_selectors:
        found = soup.find_all(tag, class_=cls)
        if found:
            return found
    return []


def extract_title_tag(item):
    """Знаходить тег із заголовком: спочатку <a class=*title*>, потім h2/h3/a."""
    return (
        item.find('a', class_=lambda c: c and 'title' in c)
        or item.find('h2')
        or item.find('h3')
        or item.find('a')
    )


def extract_title(title_tag) -> str:
    """Витягує текст заголовка."""
    return title_tag.get_text(strip=True) if title_tag else ''


def extract_url(item, title_tag) -> str:
    """Нормалізує href → абсолютний URL."""
    href = title_tag.get('href', '') or (
        item.find('a').get('href', '') if item.find('a') else ''
    )
    if not href:
        return ''
    return href if href.startswith('http') else BASE_URL + href


def extract_category(item) -> str:
    """Витягує категорію/рубрику новини."""
    cat_tag = item.find(
        lambda t: t.name in ('span', 'div', 'a')
        and any(
            'categor' in c or 'rubric' in c or 'section' in c
            for c in (t.get('class') or [])
        )
    )
    return cat_tag.get_text(strip=True) if cat_tag else ''


def extract_description(item, title: str = '') -> str:
    """Витягує лід-абзац, уникаючи дублювання заголовка."""
    desc_tag = item.find('p') or item.find(
        lambda t: t.name == 'div'
        and any(
            k in ' '.join(t.get('class') or [])
            for k in ('desc', 'lead', 'anons', 'text')
        )
    )
    description = desc_tag.get_text(strip=True) if desc_tag else ''
    return '' if description == title else description


def extract_datetime(item) -> str:
    """Витягує datetime з <time datetime=...> або текстового тегу."""
    time_tag = item.find('time')
    if time_tag:
        return (time_tag.get('datetime', '') or time_tag.get_text(strip=True))[:19]
    dt_tag = item.find(
        lambda t: any(
            k in ' '.join(t.get('class') or [])
            for k in ('date', 'time', 'ago')
        )
    )
    return dt_tag.get_text(strip=True) if dt_tag else ''


def parse_news_item(item, seen_urls: set) -> dict | None:
    """
    Парсить один контейнер новини → dict або None.
    seen_urls — множина вже доданих URL (захист від дублікатів).
    УВАГА: в threading-версії seen_urls захищений Lock (див. нижче).
    """
    title_tag = extract_title_tag(item)
    if not title_tag:
        return None

    title = extract_title(title_tag)
    if len(title) < 10:
        return None

    full_url = extract_url(item, title_tag)
    if not full_url or full_url in seen_urls:
        return None

    seen_urls.add(full_url)

    return {
        'title':       title,
        'url':         full_url,
        'category':    extract_category(item),
        'description': extract_description(item, title),
        'datetime':    extract_datetime(item),
    }


def parse_links_fallback(soup: BeautifulSoup, seen_urls: set) -> list[dict]:
    """Fallback: сканує всі <a href=/news/> якщо CSS-контейнери не знайдено."""
    news_list = []
    for a in soup.find_all('a', href=True):
        href = a.get('href', '')
        if '/news/' not in href:
            continue
        raw_text = a.get_text(separator=' ', strip=True)
        title    = re.sub(r'^\d{1,2}:\d{2}\s*', '', raw_text).strip()
        if len(title) < 10:
            continue
        full_url = href if href.startswith('http') else BASE_URL + href
        if full_url in seen_urls:
            continue
        seen_urls.add(full_url)
        time_match = re.match(r'^(\d{1,2}:\d{2})', raw_text)
        news_list.append({
            'title':       title,
            'url':         full_url,
            'category':    '',
            'description': '',
            'datetime':    time_match.group(1) if time_match else '',
        })
    return news_list


def parse_rbc_news(html: str) -> list[dict]:
    """Головна функція парсингу HTML → list[dict] новин."""
    soup      = BeautifulSoup(html, 'html.parser')
    news_list = []
    seen_urls = set()  # локальна — для одної сторінки (без потоків)

    for item in find_news_containers(soup):
        parsed = parse_news_item(item, seen_urls)
        if parsed:
            news_list.append(parsed)

    if not news_list:
        news_list = parse_links_fallback(soup, seen_urls)

    return news_list


print('✅ Функції парсингу визначено')
print(f'   Компонентів: 9 функцій + 1 головна parse_rbc_news()')


# 3. Чому Threading ефективний для Web Scraping

## IO-bound vs CPU-bound

| Тип задачі | Bottleneck | Рішення | Приклад |
| ---------- | ---------- | ------- | ------- |
| **IO-bound** | Очікування мережі/файлу | `threading` / `asyncio` | Web scraping |
| **CPU-bound** | Обчислення | `multiprocessing` | ML training, image resize |

## Профіль часу Web Scraping запиту

```
Один requests.get(url) ≈ 2 секунди:

[0.001с] DNS resolution
[0.050с] TCP 3-way handshake
[0.080с] TLS handshake
[0.001с] HTTP request formulation
[1.860с] ⏳ WAITING FOR SERVER RESPONSE  ← 93% часу
[0.005с] HTTP response parsing
[0.003с] BeautifulSoup parse HTML
────────
[2.000с] Разом
```

**93% часу Python нічого не робить** — просто чекає байти від сервера.

## GIL та IO: чому немає проблем

```mermaid
sequenceDiagram
    participant GIL as 🔒 GIL
    participant T1 as Thread 1 (rbc.ua/)
    participant T2 as Thread 2 (rbc.ua/news/)
    participant NET as 🌐 Network

    T1->>GIL: acquire()
    T1->>NET: TCP connect + send HTTP
    T1->>GIL: release() ← відпускає під час очікування!

    T2->>GIL: acquire() ✅ (T1 не тримає!)
    T2->>NET: TCP connect + send HTTP
    T2->>GIL: release()

    Note over T1,T2: Обидва чекають відповідь паралельно

    NET-->>T1: HTML відповідь
    T1->>GIL: acquire() → parse HTML
    NET-->>T2: HTML відповідь
```

**Висновок:** GIL відпускається під час `socket.recv()`. Threading для IO = справжній паралелізм.


# 4. requests.Session — Connection Pooling

## Чому Session краще за окремі get()

| | `requests.get()` (без Session) | `Session.get()` |
| --- | --- | --- |
| TCP з'єднання | Нове кожного разу | **Повторно використовує** |
| TLS handshake | Кожного разу | Один раз на хост |
| Headers | Потрібно передавати кожного разу | Зберігаються в Session |
| Thread-safety | N/A | Session NOT thread-safe — **одна Session на потік!** |

**Важливо:** `requests.Session` **не є thread-safe**.  
Тому ми передаємо об'єкт Session у кожен потік окремо  
або створюємо нову Session всередині `fetch_page()`.


In [ ]:
# === fetch_page: ОДИНИЦЯ РОБОТИ ДЛЯ ОДНОГО ПОТОКУ ===

def fetch_page(url: str) -> dict:
    """
    Завантажує одну сторінку та парсить новини.

    Кожен потік створює власну Session — requests.Session не є thread-safe.
    Повертає dict з полями:
        url    : str        — вхідний URL
        news   : list[dict] — список спарсованих новин
        elapsed: float      — час завантаження у секундах
        error  : str | None — текст помилки або None
    """
    t_start = time.perf_counter()

    # Власна Session на потік — не ділимо з іншими потоками
    with requests.Session() as session:
        session.headers.update(HEADERS)

        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()  # кидає HTTPError для 4xx/5xx

            news = parse_rbc_news(response.text)

            return {
                'url':     url,
                'news':    news,
                'elapsed': time.perf_counter() - t_start,
                'error':   None,
            }

        except requests.exceptions.Timeout:
            return {'url': url, 'news': [], 'elapsed': time.perf_counter() - t_start,
                    'error': f'Timeout ({REQUEST_TIMEOUT}с)'}

        except requests.exceptions.HTTPError as e:
            return {'url': url, 'news': [], 'elapsed': time.perf_counter() - t_start,
                    'error': f'HTTP {e.response.status_code}'}

        except requests.exceptions.ConnectionError as e:
            return {'url': url, 'news': [], 'elapsed': time.perf_counter() - t_start,
                    'error': f'ConnectionError: {str(e)[:60]}'}

        except Exception as e:
            return {'url': url, 'news': [], 'elapsed': time.perf_counter() - t_start,
                    'error': f'Unexpected: {str(e)[:60]}'}


print('✅ fetch_page() визначено')
print('   Одна Session на потік | timeout | raise_for_status | except chain')


# 5. Baseline: Sequential (для benchmark)

Спочатку запускаємо **послідовне** завантаження — отримаємо `time_sequential`.  
Потім порівняємо з threading. Без baseline benchmark безглуздий.


In [ ]:
# === SEQUENTIAL BASELINE з tqdm ===

print('⏳ Sequential завантаження...')
print('=' * 55)

seq_results = []
t_seq_start = time.perf_counter()

# tqdm обгортає список URL — бачимо прогрес і ETA
for url in tqdm(PAGES_TO_SCRAPE, desc='Sequential', unit='page', colour='red'):
    result = fetch_page(url)

    status = f'❌ {result["error"]}' if result['error'] else \
             f'✅ {len(result["news"]):3d} новин  {result["elapsed"]:.2f}с'
    tqdm.write(f'  {url.split("/")[-2] or "home":12s}  {status}')

    seq_results.append(result)

time_sequential = time.perf_counter() - t_seq_start

seq_total = sum(len(r['news']) for r in seq_results)
print('=' * 55)
print(f'⏱  Sequential: {time_sequential:.2f}с  |  Новин: {seq_total}')


# 6. ThreadPoolExecutor: Concurrent Scraping

## Архітектура

```python
# Крок 1: Відправляємо всі задачі одночасно
future_to_url = {
    executor.submit(fetch_page, url): url   # submit повертає Future негайно
    for url in PAGES_TO_SCRAPE
}

# Крок 2: Обробляємо результати по мірі завершення
for future in as_completed(future_to_url):
    result = future.result()  # блокується тільки на цьому future
    # обробляємо result...
```

## as_completed vs executor.map

| | `executor.map` | `as_completed` |
| --- | --- | --- |
| Порядок результатів | Зберігає порядок входу | Порядок завершення |
| Блокування | На ПЕРШОМУ незавершеному | Тільки на поточному |
| Якщо URL 1 повільний | Чекаємо перш ніж обробити 2, 3 | 2 і 3 обробляються одразу |
| Коли використовувати | Потрібен порядок результатів | Максимальна швидкість |

## Thread-safe дедуплікація

Різні потоки можуть знайти одну й ту саму URL на різних сторінках.  
Спільний `seen_urls` set + `threading.Lock` → атомарна перевірка та додавання.


In [ ]:
# === CONCURRENT SCRAPER: ThreadPoolExecutor + as_completed + tqdm ===

print('🚀 Concurrent завантаження...')
print('=' * 55)

global_seen_urls: set[str] = set()
seen_lock = threading.Lock()

all_news: list[dict] = []
concurrent_results: list[dict] = []

t_conc_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    # submit() повертає Future НЕГАЙНО — всі запити стартують одночасно
    future_to_url = {
        executor.submit(fetch_page, url): url
        for url in PAGES_TO_SCRAPE
    }

    # tqdm обгортає as_completed:
    #   total=  → знає скільки всього futures
    #   desc=   → підпис зліва від progress bar
    #   unit=   → одиниці (показує 'page/s')
    #   colour= → колір bar у терміналі
    progress = tqdm(
        as_completed(future_to_url),
        total=len(future_to_url),
        desc='Concurrent',
        unit='page',
        colour='green',
    )

    for future in progress:
        url = future_to_url[future]
        short = url.split('/')[-2] or 'home'

        try:
            result = future.result()
        except Exception as e:
            # tqdm.write() друкує ПОЗА progress bar — не ламає вигляд
            tqdm.write(f'  ❌ {short}: Uncaught: {e}')
            continue

        concurrent_results.append(result)

        if result['error']:
            tqdm.write(f'  ❌ {short:12s}  {result["error"]}')
            continue

        new_count = 0
        for item in result['news']:
            with seen_lock:
                if item['url'] not in global_seen_urls and item['title']:
                    global_seen_urls.add(item['url'])
                    all_news.append(item)
                    new_count += 1

        # tqdm.write → рядок вище progress bar (не скролить його)
        tqdm.write(
            f'  ✅ {short:12s}  '
            f'{new_count:3d} нових  {result["elapsed"]:.2f}с  '
            f'(всього: {len(all_news)})'
        )

        # Оновлюємо postfix progress bar → показує live статистику
        progress.set_postfix({
            'news': len(all_news),
            'last': f'{result["elapsed"]:.1f}s',
        })

time_concurrent = time.perf_counter() - t_conc_start

print('=' * 55)
print(f'⚡ Concurrent: {time_concurrent:.2f}с  |  Новин: {len(all_news)}')


# 7. Benchmark: Sequential vs Concurrent


In [ ]:
# === BENCHMARK ПОРІВНЯННЯ ===

speedup = time_sequential / time_concurrent if time_concurrent > 0 else 0
time_saved = time_sequential - time_concurrent

print('╔══════════════════════════════════════════════════╗')
print('║            BENCHMARK РЕЗУЛЬТАТИ                 ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Sequential:   {time_sequential:>6.2f}с                          ║')
print(f'║  Concurrent:   {time_concurrent:>6.2f}с                          ║')
print(f'║  Прискорення:  {speedup:>6.1f}x                          ║')
print(f'║  Заощаджено:   {time_saved:>6.2f}с ({time_saved/time_sequential*100:.0f}%)                  ║')
print('╚══════════════════════════════════════════════════╝')

# ── Візуалізація ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Графік 1: час завантаження кожної сторінки
ax1 = axes[0]
labels  = [r['url'].split('/')[-2] or 'home' for r in concurrent_results if not r['error']]
times   = [r['elapsed'] for r in concurrent_results if not r['error']]
ax1.barh(labels, times, color='steelblue', edgecolor='white')
ax1.axvline(sum(times)/len(times) if times else 0, color='red',
            linestyle='--', label='Середнє')
ax1.set_xlabel('Час завантаження (с)')
ax1.set_title('Час кожної сторінки (concurrent)')
ax1.legend()

# Графік 2: sequential vs concurrent
ax2 = axes[1]
bars = ax2.bar(['Sequential', 'Concurrent'], [time_sequential, time_concurrent],
               color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.5)
ax2.bar_label(bars, fmt='%.2fс', padding=3, fontsize=12)
ax2.set_ylabel('Час (секунди)')
ax2.set_title(f'Прискорення: {speedup:.1f}x')
ax2.set_ylim(0, max(time_sequential, time_concurrent) * 1.2)

plt.suptitle('Web Scraping: Sequential vs ThreadPoolExecutor', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## Інтерпретація результатів

**Теоретичне максимальне прискорення** для IO-bound задачі з N потоками:

```
Speedup_max = N = MAX_WORKERS

Але реальне прискорення обмежене:
  1. Найповільнішим запитом (bottleneck)
  2. Мережевою затримкою до rbc.ua
  3. Overhead на створення потоків та context switching
```

**Якщо прискорення менше очікуваного** — одна або кілька сторінок повільні  
(throttling, CDN, геолокація). Concurrent час ≈ `max(per_page_times)`, не `sum`.

**Закон Amdahl:** Прискорення обмежене послідовними частинами.  
У нашому випадку: BeautifulSoup парсинг (CPU) — послідовний, але він < 1% часу.


# 8. Результати: DataFrame та Аналіз


In [ ]:
# === КОНВЕРТУЄМО У DataFrame ===

df = pd.DataFrame(all_news)

# Fallback якщо сайт не доступний або заблокував
if df.empty or len(df) < 5:
    print('Мало новин зібрано (можливо rbc.ua недоступний або використовує JS).')
    print('Завантажуємо тестовий набір для демонстрації pipeline.\n')
    sample_news = [
        {'title': 'НБУ підвищив облікову ставку до 15% річних',
         'url': 'https://www.rbc.ua/ukr/news/nbu-1', 'category': 'Економіка',
         'description': 'Рішення набуде чинності з наступного тижня.',
         'datetime': '2024-05-08T10:30:00'},
        {'title': 'Уряд схвалив пакет допомоги для малого бізнесу',
         'url': 'https://www.rbc.ua/ukr/news/gov-1', 'category': 'Економіка',
         'description': 'Загальний обсяг фінансування — 5 млрд грн.',
         'datetime': '2024-05-08T09:15:00'},
        {'title': 'ЗСУ відбили атаку у Харківській області',
         'url': 'https://www.rbc.ua/ukr/news/zsu-1', 'category': 'Суспільство',
         'description': 'Ворог зазнав значних втрат.', 'datetime': '2024-05-08T08:45:00'},
        {'title': 'МВФ схвалив черговий транш для України на 880 млн доларів',
         'url': 'https://www.rbc.ua/ukr/news/imf-1', 'category': 'Економіка',
         'description': 'Кошти підуть на підтримку бюджету.', 'datetime': '2024-05-07T16:00:00'},
        {'title': 'Енергосистема України витримала нічні удари',
         'url': 'https://www.rbc.ua/ukr/news/energy-1', 'category': 'Суспільство',
         'description': 'Світло є у всіх регіонах.', 'datetime': '2024-05-07T14:00:00'},
        {'title': 'Нові санкції ЄС проти Росії: що потрапило до списку',
         'url': 'https://www.rbc.ua/ukr/news/sanctions-1', 'category': 'Світ',
         'description': '14-й пакет санкцій включає понад 100 компаній.', 'datetime': '2024-05-07T12:00:00'},
        {'title': 'Гривня зміцнилась до 38.5 за долар',
         'url': 'https://www.rbc.ua/ukr/news/hryvnia-1', 'category': 'Економіка',
         'description': 'НБУ проводить інтервенції на валютному ринку.', 'datetime': '2024-05-06T08:00:00'},
        {'title': 'Збірна України з футболу готується до відбору на ЧС',
         'url': 'https://www.rbc.ua/ukr/news/sport-1', 'category': 'Спорт',
         'description': 'Тренування розпочнуться цього тижня.', 'datetime': '2024-05-05T16:00:00'},
    ]
    df = pd.DataFrame(sample_news)

print(f'DataFrame: {df.shape}  (рядків × колонок)')
print(f'Колонки: {list(df.columns)}')
df.head(10)


In [ ]:
# === АНАЛІЗ ЗІБРАНИХ НОВИН з tqdm ===

print(f'Всього новин:     {len(df)}')
print(f'З категорією:     {df["category"].ne("").sum()}')
print(f'З описом:         {df["description"].ne("").sum()}')
print(f'З датою:          {df["datetime"].ne("").sum()}')

# tqdm.pandas() — progress bar для pandas .apply()
# Корисно при тисячах рядків або важких операціях (NLP, regex)
tqdm.pandas(desc='Довжина заголовків')
df['title_len'] = df['title'].progress_apply(len)

tqdm.pandas(desc='Токенізація')
df['word_count'] = df['title'].progress_apply(
    lambda t: len(re.findall(r'[а-яіїєёa-z]+', t.lower()))
)

print(f'\nСередня довжина заголовка: {df["title_len"].mean():.0f} символів')
print(f'Середня кількість слів:    {df["word_count"].mean():.1f}')

# Розподіл по категоріям
cat_counts = df['category'].replace('', 'Без категорії').value_counts()

if not cat_counts.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    cat_counts.head(10).plot(kind='barh', ax=axes[0],
                              color='steelblue', edgecolor='white')
    axes[0].set_title('Розподіл новин по категоріям')
    axes[0].set_xlabel('Кількість новин')

    df['word_count'].value_counts().sort_index().plot(
        kind='bar', ax=axes[1], color='mediumseagreen', edgecolor='white'
    )
    axes[1].set_title('Розподіл кількості слів у заголовках')
    axes[1].set_xlabel('Слів у заголовку')
    axes[1].set_ylabel('Кількість новин')

    plt.tight_layout()
    plt.show()

print('\nТоп-5 найдовших заголовків:')
for _, row in df.nlargest(5, 'title_len').iterrows():
    print(f'  [{row["title_len"]:3d} символів] {row["title"]}')


# Summary

## Що ми зробили

| Крок | Дія |
| ---- | --- |
| 1 | Розбили монолітну parse функцію на 9 чистих компонентів |
| 2 | Написали `fetch_page()` з повним error handling |
| 3 | Виміряли sequential baseline |
| 4 | Замінили for-loop на `ThreadPoolExecutor` + `as_completed` |
| 5 | Додали thread-safe дедуплікацію через `Lock` |
| 6 | Порівняли результати та візуалізували speedup |
| 7 | Додали `tqdm` до sequential, concurrent та pandas |

## Ключові принципи

```
Web scraping — IO-bound → threading ефективний
GIL відпускається під час socket.recv() → справжній паралелізм
Session на потік → не thread-safe, але ефективно
as_completed > executor.map → обробляємо результати по мірі готовності
Lock для shared state → global_seen_urls захищений від race condition
future.result() в try/except → silent failures стають видимими
```

## Production checklist для concurrent scraper

- [x] `REQUEST_TIMEOUT` на кожному запиті
- [x] `raise_for_status()` для HTTP 4xx/5xx
- [x] `try/except` навколо `future.result()`
- [x] Thread-safe `seen_urls` + `Lock`
- [x] `max_workers` ≤ кількості URL (зайві потоки = overhead)
- [x] Ввічлива ідентифікація у `User-Agent`

---

## Наступний крок: asyncio

Threading показав значне прискорення. Але при **тисячах** URL потоки стануть проблемою:
- Кожен потік ≈ 8 MB RAM
- 1000 потоків = 8 GB RAM

`asyncio` + `aiohttp` — один event loop, мільйони concurrent connections.

## tqdm: ключові патерни

```python
# Sequential loop
for url in tqdm(urls, desc='Scraping', unit='url'):
    fetch(url)

# Concurrent (найважливіший патерн)
for future in tqdm(as_completed(futures), total=len(futures)):
    result = future.result()

# Pandas (важкі apply операції)
tqdm.pandas()
df['result'] = df['col'].progress_apply(heavy_fn)

# tqdm.write() — лог поза progress bar
tqdm.write(f'❌ Error: {url}')   # не ламає вигляд bar

# set_postfix() — live статистика в bar
pbar.set_postfix({'news': 42, 'errors': 1})
```
Але це тема наступного уроку.

# Phase 2 — Article Deep Scraper

## Реальний кейс: чому threading необхідний

На попередньому кроці ми зібрали **199 посилань** — лише заголовки та URL.  
Тепер потрібно **зайти на кожну статтю** і зібрати повний контент:

```
df['url'][0] → https://www.rbc.ua/rus/news/ukrayina-spilno-...
df['url'][1] → https://www.rbc.ua/rus/news/google-meet-...
...           199 посилань
```

| Підхід | Час (199 статей × ~1.5с) |
| ------ | ------------------------- |
| Sequential | 199 × 1.5с = **~5 хвилин** |
| Threading (MAX_WORKERS=20) | max_latency ≈ **~8–15с** |

## Що ми витягуємо з кожної статті

```
fetch_article_content(url)
    ↓
    ├── lead     — лід-абзац (перший параграф)
    ├── body     — повний текст статті (перші 500 символів)
    ├── pub_date — дата публікації з <time datetime=...>
    ├── author   — автор матеріалу
    └── tags     — теги / рубрики
```

## Архітектура Phase 2

```mermaid
flowchart TD
    DF["df — 199 рядків\n(title, url)"] --> URLS["df['url'].tolist()"]
    URLS --> TPE["ThreadPoolExecutor\nmax_workers=20"]
    TPE --> |"submit × 199"| T1["Thread 1\nfetch_article(url_1)"]
    TPE --> T2["Thread 2\nfetch_article(url_2)"]
    TPE --> TN["Thread N\nfetch_article(url_N)"]
    T1 & T2 & TN --> AC["as_completed + tqdm"]
    AC --> ENRICH["df.merge(article_data)"]
    ENRICH --> DF2["df_enriched — повний контент"]
```


In [ ]:
# === PHASE 2: ФУНКЦІЯ ПАРСИНГУ ОКРЕМОЇ СТАТТІ ===

ARTICLE_WORKERS = 20   # IO-bound → більше потоків = краще (до ~30)
ARTICLE_TIMEOUT = 10   # секунд


def extract_lead(soup: BeautifulSoup) -> str:
    """Витягує лід-абзац статті."""
    for sel in [
        lambda s: s.find('div', class_=lambda c: c and 'article__header__anons' in c),
        lambda s: s.find('div', class_=lambda c: c and ('lead' in c or 'anons' in c or 'subtitle' in c)),
        lambda s: s.find('p', class_=lambda c: c and ('lead' in c or 'intro' in c)),
    ]:
        tag = sel(soup)
        if tag:
            return tag.get_text(' ', strip=True)[:300]
    return ''


def extract_body(soup: BeautifulSoup) -> str:
    """
    Витягує перші ~500 символів тіла статті.
    rbc.ua використовує клас 'publication-body' для основного контенту.
    """
    body_tag = (
        soup.find('div', class_='publication-body')         # rbc.ua основний клас
        or soup.find('div', class_=lambda c: c and 'article__content' in c)
        or soup.find('div', class_=lambda c: c and 'article-content' in c)
        or soup.find('div', class_=lambda c: c and 'content__article' in c)
        or soup.find('div', class_=lambda c: c and 'article__text' in c)
    )
    if not body_tag:
        return ''
    paragraphs = [
        p.get_text(' ', strip=True)
        for p in body_tag.find_all('p')
        if len(p.get_text(strip=True)) > 30
    ]
    return ' '.join(paragraphs)[:500]


def extract_pub_date(soup: BeautifulSoup) -> str:
    """Витягує дату публікації з <time datetime=...>."""
    time_tag = soup.find('time', attrs={'datetime': True})
    if time_tag:
        return time_tag['datetime'][:19]
    dt_tag = soup.find(lambda t: t.name in ('span', 'div')
                       and any(k in ' '.join(t.get('class') or [])
                               for k in ('date', 'time', 'publish')))
    return dt_tag.get_text(strip=True)[:20] if dt_tag else ''


def extract_author(soup: BeautifulSoup) -> str:
    """Витягує автора матеріалу."""
    for cls_kw in ('author', 'byline', 'journalist'):
        tag = soup.find(lambda t: t.name in ('span', 'div', 'a')
                        and any(cls_kw in c for c in (t.get('class') or [])))
        if tag:
            text = tag.get_text(strip=True)
            if 5 < len(text) < 60:
                return text
    return ''


def extract_tags(soup: BeautifulSoup) -> list[str]:
    """Витягує теги/рубрики статті."""
    tags = []
    for sel_cls in ('article__tags', 'tags', 'tag-list', 'rubrics', 'intext-tags'):
        container = soup.find(class_=lambda c: c and sel_cls in c)
        if container:
            tags = [a.get_text(strip=True) for a in container.find_all('a') if a.get_text(strip=True)]
            break
    return tags[:5]


def fetch_article_content(url: str) -> dict:
    """
    Завантажує окрему статтю та витягує ключову інформацію.

    Повертає dict:
        url      : str
        lead     : str  — перший абзац
        body     : str  — текст статті (до 500 символів)
        pub_date : str  — дата публікації
        author   : str  — автор
        tags     : list — теги
        elapsed  : float
        error    : str | None
    """
    t0 = time.perf_counter()
    with requests.Session() as session:
        session.headers.update(HEADERS)
        try:
            r = session.get(url, timeout=ARTICLE_TIMEOUT)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, 'html.parser')
            return {
                'url':      url,
                'lead':     extract_lead(soup),
                'body':     extract_body(soup),
                'pub_date': extract_pub_date(soup),
                'author':   extract_author(soup),
                'tags':     extract_tags(soup),
                'elapsed':  round(time.perf_counter() - t0, 3),
                'error':    None,
            }
        except requests.exceptions.Timeout:
            return {'url': url, 'lead': '', 'body': '', 'pub_date': '', 'author': '',
                    'tags': [], 'elapsed': round(time.perf_counter() - t0, 3), 'error': 'Timeout'}
        except Exception as e:
            return {'url': url, 'lead': '', 'body': '', 'pub_date': '', 'author': '',
                    'tags': [], 'elapsed': round(time.perf_counter() - t0, 3), 'error': str(e)[:60]}


print('✅ Article scraper функції визначено')
print(f'   ARTICLE_WORKERS={ARTICLE_WORKERS}  |  ARTICLE_TIMEOUT={ARTICLE_TIMEOUT}с')
print('   extract_body: шукає publication-body (rbc.ua) → article__content → article-content')


In [ ]:
# === PHASE 2: THREADING ПО ВСІХ СТАТТЯХ ===

# Беремо URL зі DataFrame (skip fallback sample URLs, беремо лише реальні rbc.ua/rus/news)
article_urls = df[df['url'].str.contains('/news/', na=False)]['url'].tolist()
# Обмежуємо до 30 статей для демонстрації (щоб не навантажувати сервер)
article_urls = article_urls[:30]

print(f'🔗 Статей для обробки: {len(article_urls)}')
print(f'⚙️  Workers: {ARTICLE_WORKERS}  |  Timeout: {ARTICLE_TIMEOUT}с')
print('=' * 60)

article_results: list[dict] = []
errors_count = 0
t_articles_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=ARTICLE_WORKERS) as executor:
    future_to_url = {
        executor.submit(fetch_article_content, url): url
        for url in article_urls
    }

    progress = tqdm(
        as_completed(future_to_url),
        total=len(future_to_url),
        desc='Статті',
        unit='art',
        colour='cyan',
    )

    for future in progress:
        try:
            result = future.result()
        except Exception as e:
            tqdm.write(f'  ❌ Uncaught: {e}')
            errors_count += 1
            continue

        if result['error']:
            tqdm.write(f'  ❌ {result["url"][-50:]}  →  {result["error"]}')
            errors_count += 1
        else:
            has_body = bool(result['body'])
            has_lead = bool(result['lead'])
            tqdm.write(
                f'  ✅ {result["elapsed"]:.2f}с  '
                f'lead={"✓" if has_lead else "—"}  '
                f'body={"✓" if has_body else "—"}  '
                f'date={result["pub_date"][:10] or "—"}  '
                f'{result["url"][-45:]}'
            )
            article_results.append(result)

        progress.set_postfix({
            'ok': len(article_results),
            'err': errors_count,
        })

t_articles_total = time.perf_counter() - t_articles_start

print('=' * 60)
print(f'⚡ Час: {t_articles_total:.2f}с  |  OK: {len(article_results)}  |  Errors: {errors_count}')
print(f'   Теоретичний sequential: {len(article_urls) * 1.5:.0f}с+')
print(f'   Реальне прискорення:    ~{len(article_urls) * 1.5 / t_articles_total:.1f}x')


In [ ]:
# === ЗБАГАЧУЄМО DataFrame + ПОКАЗУЄМО ВАЖЛИВУ ІНФОРМАЦІЮ ===

# Конвертуємо результати у DataFrame та приєднуємо до основного
df_articles = pd.DataFrame(article_results).drop(columns=['elapsed', 'error'], errors='ignore')

df_enriched = df.merge(df_articles, on='url', how='left')

# Заповнюємо пусті поля
for col in ('lead', 'body', 'pub_date', 'author'):
    if col in df_enriched.columns:
        df_enriched[col] = df_enriched[col].fillna('')
if 'tags' in df_enriched.columns:
    df_enriched['tags'] = df_enriched['tags'].apply(lambda x: x if isinstance(x, list) else [])

# Зведена статистика
enriched_count = df_enriched['body'].ne('').sum() if 'body' in df_enriched.columns else 0
print(f'DataFrame збагачено: {df_enriched.shape}')
print(f'Статей з тілом:      {enriched_count} / {len(df_enriched)}')
print(f'Статей з датою:      {df_enriched["pub_date"].ne("").sum() if "pub_date" in df_enriched.columns else 0}')
print(f'Статей з автором:    {df_enriched["author"].ne("").sum() if "author" in df_enriched.columns else 0}')
print()

# ── Топ-5 статей з найбільшим тілом ──────────────────────────────────────────
print('═' * 65)
print('ТОП СТАТЕЙ З ПОВНИМ КОНТЕНТОМ')
print('═' * 65)

if 'body' in df_enriched.columns:
    # nlargest очікує назву колонки → додаємо тимчасову '_body_len'
    top_articles = (
        df_enriched[df_enriched['body'].str.len() > 50]
        .assign(_body_len=lambda d: d['body'].str.len())
        .nlargest(5, '_body_len')
    )
    for i, (_, row) in enumerate(top_articles.iterrows(), 1):
        print(f'\n[{i}] {row["title"]}')
        if row.get('pub_date'):
            print(f'    📅 {row["pub_date"]}')
        if row.get('author'):
            print(f'    ✍️  {row["author"]}')
        if row.get('lead'):
            print(f'    💬 {row["lead"][:120]}...')
        if row.get('body'):
            print(f'    📄 {row["body"][:200]}...')
        if row.get('tags'):
            print(f'    🏷  {", ".join(row["tags"])}')

# ── Показуємо перші рядки збагаченого DataFrame ──────────────────────────────
print('\n' + '═' * 65)
cols_to_show = [c for c in ('title', 'pub_date', 'author', 'lead', 'tags') if c in df_enriched.columns]
df_enriched[cols_to_show].head(10)


# Phase 3 — NLP Analysis

## Що ми аналізуємо

Маємо `df_enriched` — 200 новин із заголовками та тілом статей.  
Тепер запустимо **NLP pipeline** на цих текстах:

```
df_enriched['title'] + df_enriched['body']
         ↓
  [1] Препроцесинг  → токенізація, стоп-слова, лематизація (pymorphy3)
         ↓
  [2] Word Frequency → Counter, FreqDist, топ-слова
         ↓
  [3] N-grams       → bigrams/trigrams: «ракетна атака», «штучний інтелект»
         ↓
  [4] Sentiment     → VADER + Ukrainian tone dictionary
```

## Бібліотеки

| Бібліотека | Роль |
| ---------- | ---- |
| `nltk` | Токенізація, стоп-слова, частотний аналіз, VADER |
| `pymorphy3` | Морфологічний аналізатор — приводить слово до початкової форми (лема) |
| `Ukrainian Stopwords` | Зовнішній список стоп-слів для української мови |
| `lang-uk tone-dict` | Тональний словник: слово → score від -1 до +1 |

## Чому потрібна лематизація для української мови

```
«дрон», «дрона», «дронів», «дроном» → лема: «дрон»
«атакував», «атакувала», «атакують» → лема: «атакувати»

Без лематизації Counter вважає це різними словами → спотворений аналіз.
```


In [ ]:
# === NLP: ВСТАНОВЛЕННЯ ЗАЛЕЖНОСТЕЙ ТА ІМПОРТИ ===

# pymorphy3 — сучасний форк pymorphy3:
#   ✅ немає залежності від pkg_resources / setuptools
#   ✅ Ukrainian словник вбудований
#   ✅ API ідентичний pymorphy3: morph.parse(word)[0].normal_form
%pip install -q pymorphy3

import os, csv, string, re
import nltk
from collections import Counter
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pymorphy3

for resource in ('stopwords', 'punkt', 'vader_lexicon', 'punkt_tab'):
    nltk.download(resource, quiet=True)

# ── Локальний список стоп-слів (без залежності від nltk filesystem) ───────────
STOPWORDS_UK: frozenset = frozenset({
    # Сполучники
    'та', 'і', 'й', 'а', 'але', 'або', 'чи', 'що', 'як', 'бо', 'якщо',
    'то', 'вже', 'ще', 'ж', 'б', 'би', 'адже', 'хоч', 'тобто', 'проте',
    'втім', 'однак', 'тож',
    # Частки
    'не', 'ні', 'так', 'ось', 'от', 'навіть', 'саме', 'лише', 'тільки',
    'просто', 'зовсім', 'майже', 'трохи', 'дуже', 'знову',
    'ніколи', 'завжди', 'іноді', 'нарешті', 'звісно', 'справді',
    # Прийменники
    'від', 'для', 'за', 'під', 'над', 'між', 'через', 'про', 'при',
    'без', 'після', 'перед', 'серед', 'біля', 'щодо', 'понад',
    # Займенники (леми)
    'він', 'вона', 'воно', 'вони', 'ми', 'ви', 'я', 'ти', 'це', 'те',
    'той', 'ці', 'цей', 'ця', 'хто', 'де', 'коли', 'куди', 'навіщо',
    'який', 'свій', 'мій', 'його', 'її', 'їх', 'їм', 'їй',
    'мені', 'тобі', 'йому', 'вам', 'нам', 'мене', 'тебе', 'себе',
    # Дієслова-зв'язки
    'бути', 'мати', 'може', 'можна', 'треба', 'стати',
    'сказати', 'заявити', 'повідомити', 'розповісти', 'зазначити',
    # Міри та числа
    'один', 'два', 'три', 'пів', 'млн', 'млрд', 'грн', 'тис',
    # Часові
    'сьогодні', 'вчора', 'завтра', 'зараз', 'тепер', 'потім', 'тоді',
    # Загальні без змісту
    'тут', 'там', 'більше', 'менше', 'багато', 'мало', 'всі', 'все',
    'всього', 'всіх', 'сам', 'інший', 'такий',
    'відео', 'фото', 'читати', 'дивитися', 'детальніше',
    # Англійські (трапляються в текстах)
    'the', 'and', 'for', 'with',
} | set(string.punctuation) | {'«', '»', '—', '–', '"', "'", '`'})

print(f'Стоп-слів: {len(STOPWORDS_UK)}')

# ── Тональний словник lang-uk ─────────────────────────────────────────────────
print('Завантажуємо tone dictionary...')
_r2 = requests.get(
    'https://raw.githubusercontent.com/lang-uk/tone-dict-uk/master/tone-dict-uk.tsv',
    timeout=10
)
TONE_DICT: dict[str, float] = {}
for row in csv.reader(_r2.text.splitlines(), delimiter='\t'):
    if len(row) == 2:
        try:
            TONE_DICT[row[0]] = float(row[1])
        except ValueError:
            pass
print(f'  Тональних слів: {len(TONE_DICT)}')

# ── Морфологічний аналізатор (pymorphy3) ──────────────────────────────────────
MORPH = pymorphy3.MorphAnalyzer(lang='uk')

demo_words = ['трампа', 'трамп', 'україну', 'україни', 'дронів', 'дрон', 'лише']
print('\nДемонстрація лематизації (pymorphy3):')
for w in demo_words:
    lemma = MORPH.parse(w)[0].normal_form
    mark = '→ стоп' if lemma in STOPWORDS_UK else ''
    print(f'  {w:<15} → {lemma:<15} {mark}')

print('\n✅ NLP готово  |  pymorphy3:', pymorphy3.__version__)


In [ ]:
# === NLP PIPELINE: ПРЕПРОЦЕСИНГ ===

# re.findall замість word_tokenize(language='ukrainian'):
# nltk не має punkt-моделі для UK → LookupError.
# re.findall одразу токенізує + фільтрує (тільки літери ≥ 3 символи).


def lemmatize_uk(token: str) -> str:
    return MORPH.parse(token)[0].normal_form


def preprocess_uk(text: str) -> list[str]:
    """Токенізація → лематизація → видалення стоп-лем."""
    if not text or not isinstance(text, str):
        return []
    tokens = re.findall(r'[а-яіїєёa-z]{3,}', text.lower())
    return [
        lemma for lemma in (lemmatize_uk(t) for t in tokens)
        if lemma not in STOPWORDS_UK and len(lemma) > 2
    ]


print('Препроцесинг заголовків...')
tqdm.pandas(desc='Леми заголовків')
df_enriched['tokens_title'] = df_enriched['title'].progress_apply(preprocess_uk)

print('Препроцесинг body...')
tqdm.pandas(desc='Леми body')
df_enriched['tokens_body'] = df_enriched['body'].fillna('').progress_apply(preprocess_uk)

df_enriched['tokens_all'] = df_enriched['tokens_title'] + df_enriched['tokens_body']

all_flat = [t for toks in df_enriched['tokens_all'] for t in toks]
freq_check = Counter(all_flat)

print('\nВерифікація (всі форми → одна лема):')
for name in ['трамп', 'україна', 'дрон', 'атака', 'росія', 'переговори']:
    print(f'  «{name}» — {freq_check.get(name, 0)} вживань')

print(f'\nКорпус: {sum(len(t) for t in df_enriched["tokens_all"]):,} токенів  |  '
      f'Унікальних лем: {len(freq_check):,}')


In [ ]:
# === WORD FREQUENCY + N-GRAMS АНАЛІЗ ===

from nltk import bigrams as nltk_bigrams, trigrams as nltk_trigrams

# Об'єднуємо всі токени в один плоский список
all_tokens: list[str] = [t for tokens in df_enriched['tokens_all'] for t in tokens]

# ── 1. Word Frequency (Counter) ───────────────────────────────────────────────
freq = Counter(all_tokens)
top30 = freq.most_common(30)

print(f'Унікальних слів (лем): {len(freq)}')
print(f'Топ-10 слів:')
for word, count in top30[:10]:
    bar = '█' * min(count, 40)
    print(f'  {word:<20} {count:>4}  {bar}')

# ── 2. Bigrams — найчастіші двословні фрази ──────────────────────────────────
bg_counter = Counter(nltk_bigrams(all_tokens))
top_bigrams = bg_counter.most_common(15)

print('\nТоп-10 bigrams (двословні фрази):')
for (w1, w2), count in top_bigrams[:10]:
    print(f'  "{w1} {w2}"  → {count}')

# ── 3. Trigrams — трьохсловні фрази ──────────────────────────────────────────
tg_counter = Counter(nltk_trigrams(all_tokens))
top_trigrams = tg_counter.most_common(10)

print('\nТоп-5 trigrams (трьохсловні фрази):')
for (w1, w2, w3), count in top_trigrams[:5]:
    print(f'  "{w1} {w2} {w3}"  → {count}')

# ── 4. Візуалізація ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Топ-20 слів
words20   = [w for w, _ in top30[:20]]
counts20  = [c for _, c in top30[:20]]
axes[0].barh(words20[::-1], counts20[::-1], color='steelblue', edgecolor='white')
axes[0].set_title('Топ-20 найчастіших слів (леми)')
axes[0].set_xlabel('Кількість вживань')

# Топ-12 bigrams
bg_labels = [f'{w1} {w2}' for (w1, w2), _ in top_bigrams[:12]]
bg_vals   = [c for _, c in top_bigrams[:12]]
axes[1].barh(bg_labels[::-1], bg_vals[::-1], color='mediumseagreen', edgecolor='white')
axes[1].set_title('Топ-12 bigrams (двословні фрази)')
axes[1].set_xlabel('Кількість вживань')

plt.suptitle('NLP: Частотний аналіз новин rbc.ua', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── 5. Ключові теми-тренди ────────────────────────────────────────────────────
TOPIC_KEYWORDS = {
    'Війна/ЗСУ':   ['зсу', 'армія', 'військо', 'фронт', 'атака', 'удар', 'ракета', 'дрон', 'окупант', 'оборона'],
    'Технології':  ['штучний', 'інтелект', 'телефон', 'смартфон', 'google', 'apple', 'штучні', 'технологія', 'оновлення'],
    'Економіка':   ['гривня', 'долар', 'бюджет', 'нбу', 'ставка', 'інфляція', 'банк', 'ціна', 'економіка'],
    'Енергетика':  ['енергія', 'електрика', 'газ', 'атом', 'нафта', 'свердловина', 'генерація', 'укренерго'],
    'Дипломатія':  ['переговори', 'санкції', 'нато', 'євросоюз', 'трамп', 'зеленський', 'путін', 'угода'],
}

print('\n═' * 55)
print('ТЕМАТИЧНІ ТРЕНДИ (скільки разів зустрічається тема у всіх новинах)')
print('═' * 55)
topic_scores = {}
for topic, keywords in TOPIC_KEYWORDS.items():
    score = sum(freq.get(kw, 0) for kw in keywords)
    topic_scores[topic] = score
    bar = '█' * min(score, 50)
    print(f'  {topic:<15}  {score:>4}  {bar}')


In [ ]:
# === SENTIMENT ANALYSIS: VADER + UKRAINIAN TONE DICTIONARY ===

# ── Ініціалізація VADER із українським словником ─────────────────────────────
# VADER — правило-базований аналізатор тональності (trained on English)
# Ми його розширюємо українським tone-dict від lang-uk
SIA = SentimentIntensityAnalyzer()
SIA.lexicon.update(TONE_DICT)    # додаємо українські слова → score

print(f'VADER lexicon після злиття: {len(SIA.lexicon):,} слів')
print(f'  З них українські: {len(TONE_DICT):,}')


def sentiment_score(text: str) -> float:
    """
    Повертає compound score: від -1 (дуже негативний) до +1 (дуже позитивний).
    0 = нейтральний.

    Два режими:
      RAW   — аналіз оригінального тексту
      NORM  — аналіз нормалізованих лем (pymorphy3 → словник краще спрацює)
    """
    if not text:
        return 0.0
    # Нормалізуємо через pipeline
    lemmas = preprocess_uk(text)
    norm_text = ' '.join(lemmas)
    raw_score  = SIA.polarity_scores(text)['compound']
    norm_score = SIA.polarity_scores(norm_text)['compound']
    # Беремо сильніший сигнал: якщо обидва не нуль — середнє, інакше той що є
    if raw_score != 0 and norm_score != 0:
        return round((raw_score + norm_score) / 2, 4)
    return round(raw_score or norm_score, 4)


def sentiment_label(score: float) -> str:
    if score >=  0.05:  return 'positive'
    if score <= -0.05:  return 'negative'
    return 'neutral'


# ── Застосовуємо до всіх новин (заголовок + lead) ────────────────────────────
print('\nРахуємо sentiment для всіх новин...')
tqdm.pandas(desc='Sentiment')
df_enriched['sentiment_score'] = (
    (df_enriched['title'] + ' ' + df_enriched['lead'].fillna(''))
    .progress_apply(sentiment_score)
)
df_enriched['sentiment_label'] = df_enriched['sentiment_score'].apply(sentiment_label)

# ── Статистика ────────────────────────────────────────────────────────────────
label_counts = df_enriched['sentiment_label'].value_counts()
avg_score    = df_enriched['sentiment_score'].mean()

print(f'\nЗагальний тон стрічки:')
for label, count in label_counts.items():
    pct = count / len(df_enriched) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<10} {count:>4}  ({pct:.0f}%)  {bar}')
print(f'  Середній compound score: {avg_score:+.4f}')

# ── Топ позитивних та негативних новин ───────────────────────────────────────
print('\nТОП-3 ПОЗИТИВНИХ НОВИНИ:')
for _, row in df_enriched.nlargest(3, 'sentiment_score').iterrows():
    print(f'  [{row["sentiment_score"]:+.3f}] {row["title"]}')

print('\nТОП-3 НЕГАТИВНИХ НОВИНИ:')
for _, row in df_enriched.nsmallest(3, 'sentiment_score').iterrows():
    print(f'  [{row["sentiment_score"]:+.3f}] {row["title"]}')

# ── Візуалізація ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Pie chart: розподіл тональності
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
pie_colors = [colors.get(l, 'gray') for l in label_counts.index]
axes[0].pie(label_counts.values, labels=label_counts.index,
            colors=pie_colors, autopct='%1.0f%%', startangle=90)
axes[0].set_title('Розподіл тональності новин')

# Histogram: розподіл compound score
axes[1].hist(df_enriched['sentiment_score'], bins=30,
             color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='neutral')
axes[1].axvline(avg_score, color='orange', linestyle='--', linewidth=1.5,
                label=f'mean={avg_score:+.3f}')
axes[1].set_xlabel('Compound Score (-1 = негатив, +1 = позитив)')
axes[1].set_ylabel('Кількість новин')
axes[1].set_title('Розподіл sentiment scores')
axes[1].legend()

# Bar: sentiment по тематичних блоках (топ-слова кожної теми)
topic_sentiments = {}
for topic, keywords in TOPIC_KEYWORDS.items():
    mask = df_enriched['tokens_all'].apply(
        lambda toks: any(kw in toks for kw in keywords)
    )
    if mask.sum() > 0:
        topic_sentiments[topic] = df_enriched.loc[mask, 'sentiment_score'].mean()

if topic_sentiments:
    topics = list(topic_sentiments.keys())
    scores = list(topic_sentiments.values())
    bar_colors = ['#e74c3c' if s < 0 else '#2ecc71' for s in scores]
    axes[2].barh(topics, scores, color=bar_colors, edgecolor='white')
    axes[2].axvline(0, color='black', linewidth=0.8)
    axes[2].set_xlabel('Середній sentiment score')
    axes[2].set_title('Середній тон по темах')

plt.suptitle('Sentiment Analysis: Новини rbc.ua', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── Підсумкова таблиця ────────────────────────────────────────────────────────
print('\nПерші 10 новин із sentiment scores:')
df_enriched[['title', 'sentiment_score', 'sentiment_label']].head(10)


---
# Підсумок уроку: що і чому ми зробили

---

## 1. Проблема, яку вирішував threading

Web scraping — **IO-bound задача**: Python відправляє HTTP-запит і **чекає** відповіді від сервера.  
Під час очікування CPU нічого не робить — це і є проблема sequential підходу:

```
Sequential (8 сторінок × ~0.6с):
  Thread 1:  [GET url1] ──⏳── [parse] [GET url2] ──⏳── [parse] ...
                               ↑ CPU простоює

Concurrent (8 запитів одночасно):
  Thread 1:  [GET url1] ──────⏳──────[parse]
  Thread 2:  [GET url2] ───⏳──[parse]
  Thread 3:  [GET url3] ────────⏳────[parse]
             ↑ Всі чекають паралельно → час = max(), не sum()
```

---

## 2. Чому GIL не заважав

GIL **відпускається** під час `socket.recv()` — тобто під час очікування мережі.  
Поки Thread 1 чекає відповідь від rbc.ua — Thread 2 вже виконує Python-код (parse HTML).

```
Thread 1:  [py]→[socket.recv() ⏳]→[py: parse]
Thread 2:          [socket.recv() ⏳]→[py: parse]
                    ↑ Thread 2 захопив GIL поки Thread 1 чекає
```

**Правило:** IO-bound → `threading`. CPU-bound → `multiprocessing`.

---

## 3. Ключові інструменти і чому саме вони

| Інструмент | Навіщо | Альтернатива |
| ---------- | ------- | ------------ |
| `ThreadPoolExecutor` | Автоматично керує пулом потоків | `threading.Thread` (вручну) |
| `as_completed()` | Обробляємо результат одразу як готовий | `executor.map()` (чекає всіх) |
| `threading.Lock` | Захищає `seen_urls` від race condition | `queue.Queue` |
| `requests.Session` | Connection pooling — не рве TCP щоразу | `requests.get()` |
| `tqdm(as_completed())` | Live прогрес + ETA для concurrent loops | `print()` |

---

## 4. Race Condition і Thread Safety

```python
# Без Lock — два потоки можуть додати один і той же URL двічі:
if url not in seen_urls:   # Thread 1 перевіряє → True
    seen_urls.add(url)     # Thread 2 теж перевіряє → True → дублікат!

# З Lock — атомарна операція:
with seen_lock:            # Thread 2 чекає поки Thread 1 вийде з блоку
    if url not in seen_urls:
        seen_urls.add(url)
```

---

## 5. NLP Pipeline — чому такий порядок

```
re.findall(r'[а-яіїєё]{3,}')  →  токенізація без punkt-моделі
       ↓
pymorphy3.parse()[0].normal_form  →  лематизація
       ↓                               «трампа» → «трамп»
STOPWORDS_UK фільтр (по лемі)  →      «лише», «але» → видалено
       ↓
Counter / bigrams / trigrams   →  частотний аналіз
       ↓
VADER + tone-dict-uk           →  sentiment -1..+1
```

**Ключовий порядок:** лематизація → стоп-слова (не навпаки).  
Якщо фільтрувати спочатку: «трампа» пройде фільтр у неправильній формі  
і рахуватиметься окремо від «трамп».

---

## 6. Результати нашого pipeline

| Фаза | Що зробили | Результат |
| ---- | ---------- | --------- |
| Phase 1: List Scraping | 8 URL через ThreadPoolExecutor | 199 новин за ~1.3с (vs 4.8с sequential) |
| Phase 2: Article Scraping | 30 статей, 20 workers | body + author + pub_date |
| Phase 3: NLP | re.findall → pymorphy3 → Counter | топ-слова, bigrams, trigrams |
| Sentiment | VADER + 3,442 укр. слів | 54% neutral, 33% negative, 13% positive |

**Тон стрічки: -0.08** — легкий негатив. Це типово для новинних медіа.

---

## 7. Коли НЕ використовувати threading

| Сценарій | Проблема | Рішення |
| -------- | -------- | ------- |
| ML training, CV, crypto | CPU-bound, GIL блокує | `multiprocessing` |
| 1000+ з'єднань | Кожен потік ~8 MB RAM | `asyncio` + `aiohttp` |
| Одна задача без IO | Overhead > виграш | Звичайний for-loop |

---

## 8. Закон Amdahl — межа прискорення

```
Speedup = 1 / (S + P/N)

S = 0.05  (BeautifulSoup парсинг — послідовний, ~5% часу)
P = 0.95  (socket.recv — паралельний, ~95% часу)
N = 8     (кількість потоків)

При N=8:  Speedup_max = 1 / (0.05 + 0.95/8) ≈ 6.2x
При N=∞:  Speedup_max = 1 / 0.05 = 20x  ← абсолютна стеля

Ми отримали ~3.7x — реальний результат
(мережева нерівномірність + CDN throttling + TLS overhead).
```


In [ ]:
# === ФІНАЛЬНА ЗВЕДЕНА ТАБЛИЦЯ ===
# Запускай після всіх попередніх cells.
# Якщо якась фаза не виконана — показує 'N/A'.
import textwrap

# ── Збираємо значення захищено (якщо фаза не запускалась → N/A) ──────────────
_g = globals()

def _v(name, fmt='{:.2f}', fallback='N/A'):
    """Повертає відформатоване значення або 'N/A'."""
    val = _g.get(name)
    if val is None:
        return fallback
    try:
        return fmt.format(val) if '{}' in fmt or '{:' in fmt else str(val)
    except Exception:
        return str(val)

t_seq  = _g.get('time_sequential', 0)
t_con  = _g.get('time_concurrent',  1)  # 1 щоб уникнути ZeroDivisionError
speedup_val = f'{t_seq / t_con:.1f}x' if t_seq and t_con else 'N/A'

df_e = _g.get('df_enriched')
fc   = _g.get('freq_check', {})
news = _g.get('all_news', [])
arts = _g.get('article_results', [])
sd   = _g.get('STOPWORDS_UK', set())
td   = _g.get('TONE_DICT', {})
mw   = _g.get('MAX_WORKERS', '?')
aw   = _g.get('ARTICLE_WORKERS', '?')
at   = _g.get('ARTICLE_TIMEOUT', '?')

if df_e is not None:
    total_tokens = sum(len(t) for t in df_e.get('tokens_all', [[]]))
    n_neu = (df_e.get('sentiment_label', '') == 'neutral').sum()
    n_neg = (df_e.get('sentiment_label', '') == 'negative').sum()
    n_pos = (df_e.get('sentiment_label', '') == 'positive').sum()
    n_tot = len(df_e)
    avg_s = df_e.get('sentiment_score', [0]).mean() if 'sentiment_score' in df_e.columns else 0
    has_body = df_e['body'].ne('').sum() if 'body' in df_e.columns else 0
    has_date = df_e['pub_date'].ne('').sum() if 'pub_date' in df_e.columns else 0
    has_auth = df_e['author'].ne('').sum() if 'author' in df_e.columns else 0
else:
    total_tokens = n_neu = n_neg = n_pos = n_tot = has_body = has_date = has_auth = 0
    avg_s = 0

top3 = ' | '.join(w for w, _ in list(fc.items())[:3]) if fc else 'N/A'

# ── Виводимо таблицю ─────────────────────────────────────────────────────────
W = 68
HL = '\u2550' * W
HS = '\u2500' * W
print('\u2554' + HL + '\u2557')
print('\u2551' + ' ПІДСУМОК: MULTITHREADED NEWS SCRAPER + NLP PIPELINE '.center(W) + '\u2551')
print('\u2560' + HL + '\u2563')

sections = [
    ('THREADING BENCHMARK', [
        f'Sequential:   {_v("time_sequential")}с  (8 URL один за одним)',
        f'Concurrent:   {_v("time_concurrent")}с  ({mw} workers паралельно)',
        f'Прискорення:  {speedup_val}  (теоретичний max ~6x за Amdahl)',
        f'Новин зібрано: {len(news)}  |  Дублікати видалено через Lock',
    ]),
    ('ARTICLE DEEP SCRAPE — Phase 2', [
        f'Статей оброблено: {len(arts)}  ({aw} workers  |  timeout {at}с)',
        f'З body: {has_body}  |  З датою: {has_date}  |  З автором: {has_auth}',
    ]),
    ('NLP PIPELINE — Phase 3', [
        f'Токенів (лем): {total_tokens:,}  |  Унікальних: {len(fc):,}',
        f'Стоп-слів відфільтровано: {len(sd)}',
        f'Тональний словник lang-uk: {len(td):,} слів',
        f'Топ-3 слова: {top3}',
    ]),
    ('SENTIMENT ANALYSIS', [
        f'Середній score: {avg_s:+.4f}  (негатив типовий для новин)',
        f'Neutral:  {n_neu} ({n_neu/n_tot*100:.0f}%)' if n_tot else 'Neutral:  N/A',
        f'Negative: {n_neg} ({n_neg/n_tot*100:.0f}%)' if n_tot else 'Negative: N/A',
        f'Positive: {n_pos} ({n_pos/n_tot*100:.0f}%)' if n_tot else 'Positive: N/A',
    ]),
    ('КЛЮЧОВІ УРОКИ', [
        'IO-bound = threading  (GIL вивільняється при socket.recv)',
        'as_completed > executor.map — не чекаємо повільних',
        'Lock — атомарний захист shared state від race condition',
        'Лема ДО стоп-слів — трамп/трампа стають одним словом',
        're.findall замість punkt — немає UK punkt-моделі в nltk',
        'pymorphy3 замість pymorphy2 — немає pkg_resources бага',
    ]),
]

for title, items in sections:
    print('\u2560' + HL + '\u2563')
    print('\u2551  ' + title.ljust(W - 2) + '\u2551')
    print('\u2551' + HS + '\u2551')
    for item in items:
        for i, line in enumerate(textwrap.wrap(item, width=W - 5)):
            prefix = '  \u2022 ' if i == 0 else '    '
            print('\u2551' + prefix + line.ljust(W - len(prefix)) + '\u2551')

print('\u255a' + HL + '\u255d')
print('\n\u2705 Урок 32 завершено.')
